# CS 5588 — RAG with LangChain, Chroma, and Gemini Free API
_Generated: 2025-09-14T13:53:05_

### 1) Install

In [8]:

!pip -q install -U langchain langchain-community chromadb pypdf             sentence-transformers transformers tiktoken             langchain-google-genai google-genai
print("If upgraded core libs, consider restarting runtime.")


If upgraded core libs, consider restarting runtime.


### 2) Keys & Imports

In [9]:

import os, getpass, json, sys, platform, pathlib, datetime, importlib
if not os.getenv("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass.getpass("Enter your GEMINI_API_KEY: ")
os.environ["GOOGLE_API_KEY"] = os.environ.get("GOOGLE_API_KEY", os.environ["GEMINI_API_KEY"])

from google import genai
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain.chains import RetrievalQA

pathlib.Path("data").mkdir(exist_ok=True)
pathlib.Path("artifacts").mkdir(exist_ok=True)
print("Env ready.")


Env ready.


### 3) Log environment → env_rag.json

In [10]:

def pv(m):
    try:
        mod = importlib.import_module(m)
        return getattr(mod, "__version__", "unknown")
    except: return "not installed"
env = {
  "timestamp": datetime.datetime.now().isoformat(),
  "python": sys.version, "platform": platform.platform(),
  "packages": {m: pv(m) for m in [
    "langchain","langchain_community","chromadb","tiktoken","transformers",
    "sentence_transformers","langchain_google_genai","google.genai"
  ]}
}
with open("env_rag.json","w") as f: json.dump(env, f, indent=2)
print(json.dumps(env, indent=2))


{
  "timestamp": "2025-09-19T02:47:30.296935",
  "python": "3.12.11 (main, Jun  4 2025, 08:56:18) [GCC 11.4.0]",
  "platform": "Linux-6.1.123+-x86_64-with-glibc2.35",
  "packages": {
    "langchain": "0.3.27",
    "langchain_community": "0.3.29",
    "chromadb": "1.1.0",
    "tiktoken": "0.11.0",
    "transformers": "4.56.1",
    "sentence_transformers": "5.1.0",
    "langchain_google_genai": "unknown",
    "google.genai": "1.38.0"
  }
}


### 4) Upload documents

In [11]:

try:
    from google.colab import files
    up = files.upload()
    import os
    os.makedirs("data", exist_ok=True)
    for n,c in up.items():
        open(os.path.join("data", n), "wb").write(c)
    print("Uploaded:", list(up.keys()))
except Exception as e:
    print("Colab upload UI not available.", e)


Saving paper1.pdf to paper1 (1).pdf
Saving paper2.pdf to paper2 (1).pdf
Saving paper3.pdf to paper3 (1).pdf
Uploaded: ['paper1 (1).pdf', 'paper2 (1).pdf', 'paper3 (1).pdf']


### 5) Load & chunk

In [12]:

import os
def load_docs(folder="data"):
    docs=[]
    for fname in os.listdir(folder):
        p=os.path.join(folder,fname)
        if not os.path.isfile(p): continue
        ext=fname.lower().split(".")[-1]
        try:
            if ext=="pdf": loader=PyPDFLoader(p)
            elif ext in ["txt","md","markdown"]: loader=TextLoader(p, encoding="utf-8")
            else:
                print("Skip", fname); continue
            docs += loader.load()
        except Exception as e:
            print("Fail", fname, e)
    return docs
raw_docs=load_docs("data")
print("Loaded", len(raw_docs))
splitter=RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
splits=splitter.split_documents(raw_docs)
print("Chunks:", len(splits))
if splits: print(splits[0].page_content[:400])
rag_run_config={"chunk_size":500,"chunk_overlap":100,"embedding_models_tested":[],"llm":None,"retriever_k":4}
import json
json.dump(rag_run_config, open("rag_run_config.json","w"), indent=2)


Loaded 194
Chunks: 1334
arXiv:2412.20138v7  [q-fin.TR]  3 Jun 2025
T A U R I C  R E S E A R C H
TradingAgents: Multi-Agents LLM Financial Trading
Framework
Yijia Xiao 1,3, Edward Sun 1,3, Di Luo 1,2, Wei Wang1,3
1University of California, Los Angeles (UCLA)
2Massachusetts Institute of Technology (MIT)
3Tauric Research*
Significant progress has been made in automated problem-solving using societies of agents pow-
ered by 


### 6) Vector DB (Chroma) + baseline embeddings

In [13]:

from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma
emb = SentenceTransformerEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vs = Chroma.from_documents(splits, embedding=emb, persist_directory="./chroma_minilm")
vs.persist()
retriever = vs.as_retriever(search_kwargs={"k":4})
print("Vector store ready.")
cfg=json.load(open("rag_run_config.json"))
cfg["embedding_models_tested"].append("sentence-transformers/all-MiniLM-L6-v2")
json.dump(cfg, open("rag_run_config.json","w"), indent=2)


Vector store ready.


### 7) RetrievalQA with Gemini

In [14]:

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.2)
qa = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=retriever, return_source_documents=True)
def ask(q):
    r=qa({"query":q})
    print("\nQ:", q); print("A:", r.get("result",""))
    print("\nSources:")
    for i,d in enumerate(r.get("source_documents",[])[:3]):
        print(f"[{i+1}] {d.metadata.get('source','?')} ::", d.page_content[:160].replace("\n"," ")+"...")
ask("What are the main findings relevant to our project domain?")



Q: What are the main findings relevant to our project domain?
A: Based on the provided context, the main finding relevant to a project domain concerning the business community or tech industry is:

*   There are **mixed reactions in the business community** regarding potential deregulation and increased innovation.
*   **Some tech executives are optimistic**, believing these changes could lead to **more spending and dealmaking**.

The other provided text describes team structures and processes (Analyst Team, Researcher Team) rather than specific market findings.

Sources:
[1] data/paper2.pdf :: creating mixed reactions in the business community . Some tech executives are optimistic about potential deregulation and increased innovation , which could lea...
[2] data/paper2.pdf :: creating mixed reactions in the business community . Some tech executives are optimistic about potential deregulation and increased innovation , which could lea...
[3] data/paper2 (1).pdf :: creating mixed reac

### 8) Mini-experiments (embedding swap & chunk sensitivity) — optional

In [15]:
print("\n🔍 Mini-experiment 1: Embedding Swap\n")

# Load alternative embedding model
emb_e5 = SentenceTransformerEmbeddings(model_name="intfloat/e5-small-v2")

# Create new vector store with E5 embeddings
vectordb_e5 = Chroma.from_documents(splits, emb_e5, persist_directory="./chroma_e5")
vectordb_e5.persist()
qa_e5 = RetrievalQA.from_chain_type(llm=llm, retriever=vectordb_e5.as_retriever(), chain_type="stuff")

# Compare MiniLM vs E5-small
print("MiniLM:", qa.run("List two GenAI techniques emphasized."))
print("E5-small:", qa_e5.run("List two GenAI techniques emphasized."))

# --- Chunk Size Sensitivity Experiment ---
print("\n🔍 Mini-experiment 2: Chunk Size Sensitivity\n")

splitter_small = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks_small = splitter_small.split_documents(raw_docs)

# New vector store with smaller chunks (still using MiniLM embeddings)
vectordb_small = Chroma.from_documents(chunks_small, emb, persist_directory="./chroma_minilm_small")
vectordb_small.persist()
qa_small = RetrievalQA.from_chain_type(llm=llm, retriever=vectordb_small.as_retriever(), chain_type="stuff")

# Compare default chunk size vs smaller chunk size
print("Default chunks:", qa.run("Summarize the course in one sentence."))
print("Smaller chunks:", qa_small.run("Summarize the course in one sentence."))

# --- Save Reproducibility Config ---
repro = {
    "embedding_models": ["sentence-transformers/all-MiniLM-L6-v2", "intfloat/e5-small-v2"],
    "chunking": [
        {"size": 500, "overlap": 100},  # Default
        {"size": 300, "overlap": 50}    # Smaller chunks
    ],
    "llm": "gemini-2.5-flash"
}

with open("rag_run_config.json", "w") as f:
    json.dump(repro, f, indent=2)

print("\n✅ Saved rag_run_config.json with experiment settings.")